# 📅 Dates, and things that have an order

**Time:** ~30 minutes &nbsp;•&nbsp; **Goal:** turn a column of text into real dates, and find the six rows that prove why you should never trust a month column somebody typed by hand.

Pollination is seasonal. Almost every interesting question about this survey has *when* in it — which month, early or late summer, 2017 or 2018. And the `Date` column that would answer all of them arrives as **text**.

## Run this first

In [ ]:
# Run this first.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)
df.columns = df.columns.str.strip()

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print(df[["Year", "Date", "Month"]].head())
print()
print("Date dtype:", df["Date"].dtype)

## Skill 1 — Text that looks like a date is not a date

`df["Date"].dtype` came back as `object` — pandas for "strings". Sorting it happens to work, because these dates are written year-first, which sorts correctly by luck. Everything else fails: you cannot ask what month it is, what day of the week it was, or how many days apart two watches were.

One function fixes it:

```python
df["Date"] = pd.to_datetime(df["Date"])
```

Once a column is `datetime64`, it grows a `.dt` accessor — the date equivalent of the `.str` you used on the headers:

| | |
|---|---|
| `.dt.year` `.dt.month` `.dt.day` | the parts, as numbers |
| `.dt.month_name()` `.dt.day_name()` | the parts, as text |
| `.dt.dayofyear` | 1–366, useful for plotting a season |

In [ ]:
# DEMO - one line, and the column wakes up
df["Date"] = pd.to_datetime(df["Date"])
print("dtype now:", df["Date"].dtype)

print()
print("first watch:", df["Date"].min().date())
print("last watch: ", df["Date"].max().date())
print("survey days:", df["Date"].nunique())
print("span:       ", (df["Date"].max() - df["Date"].min()).days, "days")

In [ ]:
# DEMO - who did the surveying, and when?
print(df["Date"].dt.day_name().value_counts())

Seven watches on a Saturday out of 1,620. These are not weekend volunteers dropping by — this is organised fieldwork on working days. Worth knowing before you claim the survey represents "allotment visitors".

You could not have asked that question five minutes ago.

### 📅 DIY challenge

1. Add a column `DayOfYear` using `.dt.dayofyear`. What is its range?
2. How many watches happened in each **year**? Use `.dt.year`, and check it matches the existing `Year` column.
3. `(df["Date"] - df["Date"].min())` gives you a `Timedelta`. Print `.dt.days` on it to get a plain number of days since the survey started.

> 💡 A `.dt` method with brackets — `.dt.day_name()` — returns text. Without brackets — `.dt.month` — you get a number. Both are useful; they sort very differently.

In [ ]:
# TODO - your turn

## Skill 2 — The six rows that break the `Month` column

The file already has a `Month` column, typed by hand. So why bother with dates at all?

Because now you can **check it**. Derive the month from the date, compare it to the column, and see whether they agree.

In [ ]:
# DEMO - trust, but verify
derived = df["Date"].dt.month_name()
mismatch = df[derived != df["Month"]]

print(len(mismatch), "rows where the Month column disagrees with the Date")
mismatch[["Date", "Month", "Allotment", "Plot", "Crop_Generic"]]

Six watches at Moulsecoomb, all on **1 September 2017**, all labelled August. Somebody filled in the month at the top of the sheet when the survey round began and did not change it when the round ran over into a new month. It is not a catastrophe. It is completely ordinary, and it is in every hand-entered dataset you will ever touch.

**The lesson generalises:** when a value can be *derived* and also *typed by hand*, derive it. A derived column cannot disagree with its source.

```python
df["Month"] = df["Date"].dt.month_name()   # overwrite the hand-typed one
```

In [ ]:
# DEMO - derive it, and the disagreement is gone by construction
df["Month"] = df["Date"].dt.month_name()
df["MonthNum"] = df["Date"].dt.month

print((df["Date"].dt.month_name() != df["Month"]).sum(), "mismatches now")
print()
print(df["Month"].value_counts())

### 📅 DIY challenge

1. Does the `Year` column agree with `.dt.year` everywhere? Check it the same way.
2. Now that September has six more watches, did any *other* summary change? Compare the September bumblebee total before and after the fix.
3. Come up with one other column in this file that could, in principle, be checked against another. (Hint: six species columns and one total.)

> 💡 "Check one column against another" is the cheapest quality control there is, and almost nobody does it.

In [ ]:
# TODO - your turn

## Skill 3 — Alphabetical is not chronological

Now sort the months and look at what you get.

```python
sorted(df["Month"].unique())
# ['April', 'August', 'July', 'June', 'May', 'September']
```

April, August, July, June, May, September. That is what "sorted" means for text, and it will appear on the x-axis of your first seasonal figure looking exactly this wrong.

There are two fixes, and you should know both.

**Fix 1 — sort by the number, show the name.** You already built `MonthNum`. Sort by that.

**Fix 2 — tell pandas the order.** A `Categorical` is a text column that knows its own categories, and an **ordered** Categorical knows what comes after what. Sorting then does the right thing everywhere, for good — including inside `groupby` and on a plot axis.

```python
month_order = ["April", "May", "June", "July", "August", "September"]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)
```

In [ ]:
# DEMO - the bug, in the form you will actually meet it
print("alphabetical:", sorted(df["Month"].unique()))
print()
print("value_counts, which sorts by size, not by time:")
print(df["Month"].value_counts())

In [ ]:
# DEMO - fix 2, the one worth learning
month_order = ["April", "May", "June", "July", "August", "September"]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

print(df["Month"].dtype)
print()
print("now it sorts like a season:")
print(df["Month"].value_counts().sort_index())

An ordered Categorical also lets you ask comparison questions that would be nonsense on text:

```python
df[df["Month"] > "June"]    # late summer only
```

In [ ]:
# DEMO - comparisons that only work because the column is ordered
late = df[df["Month"] > "June"]
print(len(late), "watches in July, August and September")
print(late["Month"].value_counts().sort_index())

### 📅 DIY challenge

1. Sort the whole table by `Month` and print the first and last three rows' `Date`. Are they what you expect?
2. Build a small table of total `Bumblebees` per month, in **calendar order**, using `df.groupby("Month", observed=True)["Bumblebees"].sum()`.
3. Do the same for `Total_Flowers`. Now divide one by the other — bumblebee visits per flower, month by month. Which month is the busiest *per flower*, and is it the same as the month with the most bumblebees?

> 💡 Question 3 is the whole argument of session 3 in miniature. August has the most bumblebees because August has the most watches and the most flowers.

In [ ]:
# TODO - your turn

## Skill 4 — The index can be a date 🏆

So far the index has been row numbers, which carry no meaning. Put the dates there instead and the index starts doing work:

```python
ts = df.set_index("Date")
ts.loc["2018-07"]        # every watch in July 2018
ts.loc["2018-07-01":"2018-07-15"]   # a fortnight
```

pandas understands partial date strings. `"2018"`, `"2018-07"` and `"2018-07-12"` are all valid labels for `.loc`, and each selects the right slice. This is the moment `.loc` stops looking like a clumsy alternative to `.iloc` and starts looking like the point.

In [ ]:
# DEMO - a date index, and slicing by time
ts = df.set_index("Date").sort_index()

july18 = ts.loc["2018-07"]
print("watches in July 2018:", len(july18))
print("bumblebees seen:     ", july18["Bumblebees"].sum())

print()
fortnight = ts.loc["2018-08-01":"2018-08-15"]
print("first half of August 2018:", len(fortnight), "watches")

### 🏆 DIY finale — the shape of the season

Using the date-indexed table `ts`:

1. How many watches happened in **2017** versus **2018**? Do it with `.loc`, not with the `Year` column.
2. Pull out August 2017 and August 2018 separately. Compare the number of watches, the total bumblebees, and the total flowers.
3. Bumblebee visits **per flower** in August 2017 versus August 2018 — which year was better for bumblebees? Is your answer the same if you compare raw totals instead?
4. Finally, write out the tidied table for the next session:

```python
df.to_csv("pollinators_dated.csv", index=False)
```

> 💡 Question 3 will bite. 2018 has more than twice as many watches as 2017, so any raw total is mostly measuring how much surveying was done. This is the trap that session 3 is built around — you are meeting it early on purpose.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Time series in pandas — long, but the first two sections pay for themselves: <https://pandas.pydata.org/docs/user_guide/timeseries.html>
- `ts.resample("ME")["Bumblebees"].sum()` groups by calendar month automatically once the index is a date. Try `"W"` for weeks.
- Categorical data, including why it also saves a lot of memory: <https://pandas.pydata.org/docs/user_guide/categorical.html>

## Recap

You can now:

- convert text to real dates with `pd.to_datetime()` and use the `.dt` accessor
- **check a hand-typed column against a derived one**, and prefer the derived one
- fix alphabetical month ordering with an ordered `Categorical`
- put dates in the index and slice with partial date strings like `"2018-07"`
- spot the difference between "the busiest month" and "the busiest month per flower"

**Next:** `05_groupby_basics` — one line that splits the table, summarises each piece, and puts it back together.